# Qwen Image 2.1 ConvRot signed-DoRA full-model TPU smoke

Private one-device feasibility gate. Attach the source bundle, private base/helper
asset dataset and exported paired cache. Select TPU v5e-8 and enable Internet.
The runtime must match the passing Torch/XLA 2.8.0 + libtpu 0.0.17 probe.
Only the one-rank smoke runs: no bulk training or checkpoint publication.
The original image-slider objective and cached resolution are preserved.
Dense segmented attention may fail the memory gate; failures retain diagnostics.


In [ ]:
import hashlib, importlib.metadata as metadata, json, os, pathlib, platform, subprocess, sys, uuid, zipfile
root = pathlib.Path('/kaggle/working') / ('qwen21_source_' + uuid.uuid4().hex)
root.mkdir()
inputs = pathlib.Path('/kaggle/input')
archives = list(inputs.rglob('kaggle_tpu_full_bundle.zip'))
if len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)): raise RuntimeError('Duplicate source ZIP members')
        manifest = json.loads(archive.read('source_manifest.json'))
        if set(names) != set(manifest['files']) | {'source_manifest.json'}: raise RuntimeError('Undeclared source ZIP members')
        for name, digest in manifest['files'].items():
            path = pathlib.PurePosixPath(name)
            if path.is_absolute() or '..' in path.parts or chr(92) in name: raise RuntimeError('Unsafe source path')
            content = archive.read(name)
            if hashlib.sha256(content).hexdigest() != digest: raise RuntimeError('Source hash mismatch: ' + name)
            destination = root / name
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(content)
elif not archives:
    candidates = [p for p in inputs.rglob('source_manifest.json') if json.loads(p.read_text()).get('kind') == 'qwen21_full_model_source']
    if len(candidates) != 1: raise RuntimeError('Attach exactly one full-model source bundle')
    manifest = json.loads(candidates[0].read_text())
    for name, digest in manifest['files'].items():
        path = pathlib.PurePosixPath(name)
        if path.is_absolute() or '..' in path.parts or chr(92) in name: raise RuntimeError('Unsafe source path')
        content = (candidates[0].parent / name).read_bytes()
        if hashlib.sha256(content).hexdigest() != digest: raise RuntimeError('Source hash mismatch: ' + name)
        destination = root / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)
else: raise RuntimeError('Attach exactly one full-model source bundle')
if manifest.get('schema') != 1 or manifest.get('kind') != 'qwen21_full_model_source': raise RuntimeError('Wrong source manifest')
(root / 'source_manifest.json').write_text(json.dumps(manifest, indent=2))
versions = {name: metadata.version(name) for name in ('torch', 'torch-xla', 'libtpu')}
print('Runtime:', sys.version, platform.platform(), versions)
if any(versions[name].split('+')[0] != expected for name, expected in (('torch', '2.8.0'), ('torch-xla', '2.8.0'), ('libtpu', '0.0.17'))):
    raise RuntimeError('Select the previously verified TPU image: Torch/XLA 2.8.0 + libtpu 0.0.17; this notebook never replaces runtime wheels')
if sys.version_info[:2] not in ((3, 10), (3, 11), (3, 12)): raise RuntimeError('Pinned repository dependencies require Python 3.10-3.12')
env = os.environ.copy()
for name in ('TPU_PROCESS_ADDRESSES', 'CLOUD_TPU_TASK_ID'): env.pop(name, None)
env['PJRT_DEVICE'] = 'TPU'
env['NO_ALBUMENTATIONS_UPDATE'] = '1'
preflight = subprocess.run([sys.executable, '-c', 'import torch_xla.runtime as xr; assert xr.device_type() == "TPU"; n=xr.addressable_device_count(); assert n == 8, n; print("TPU logical devices:", n)'], env=env, text=True, capture_output=True)
print(preflight.stdout)
if preflight.returncode: raise RuntimeError('TPU preflight failed: ' + preflight.stderr)
print('Verified source commit:', manifest['source_commit'], 'files:', len(manifest['files']))


In [ ]:
constraints = root / 'runtime-constraints.txt'
constraints.write_text('\n'.join(name + '==' + value for name, value in versions.items()) + '\n')
install_log = pathlib.Path('/kaggle/working/qwen21_dependency_install.log')
with install_log.open('w') as log:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--no-deps', '--index-url', 'https://download.pytorch.org/whl/cpu', 'torchvision==0.23.0+cpu', 'torchaudio==2.8.0+cpu'], stdout=log, stderr=subprocess.STDOUT, check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--constraint', str(constraints), '--requirement', str(root / 'workers/kaggle_tpu/requirements_smoke.txt')], stdout=log, stderr=subprocess.STDOUT, check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=log, stderr=subprocess.STDOUT, check=True)
after = {name: metadata.version(name) for name in versions}
if after != versions: raise RuntimeError('Dependency setup altered a protected runtime wheel')
print('Dependencies installed; log:', install_log)
cache_archives = list(inputs.rglob('paired_cache_bundle.zip'))
if len(cache_archives) > 1: raise RuntimeError('Attach exactly one paired-cache archive')
if cache_archives:
    sys.path.insert(0, str(root))
    from workers.kaggle_tpu.cache_bundle import extract_cache_bundle
    extracted_cache = extract_cache_bundle(cache_archives[0], pathlib.Path('/kaggle/working') / ('qwen21_cache_' + uuid.uuid4().hex))
    cache_manifest = extracted_cache / 'manifest.json'
cache_manifests = []
for path in inputs.rglob('manifest.json'):
    try:
        if json.loads(path.read_text()).get('kind') == 'qwen21_fizgig_paired_cache': cache_manifests.append(path)
    except (ValueError, UnicodeError): continue
asset_manifests = list(inputs.rglob('asset-manifest.json'))
if (not cache_archives and len(cache_manifests) != 1) or len(asset_manifests) != 1: raise RuntimeError('Attach exactly one paired cache and one model asset dataset')
if not cache_archives: cache_manifest = cache_manifests[0]
asset_manifest = asset_manifests[0]
print('Cache:', cache_manifest.parent, 'Assets:', asset_manifest.parent)


In [ ]:
import signal, time
report = pathlib.Path('/kaggle/working') / ('qwen21_full_smoke_' + uuid.uuid4().hex)
log_path = pathlib.Path(str(report) + '.log')
command = [sys.executable, str(root / 'workers/kaggle_tpu/full_model_smoke.py'), '--cache', str(cache_manifest.parent), '--asset-manifest', str(asset_manifest), '--output', str(report)]
timeout_minutes = 90
with log_path.open('w') as log:
    process = subprocess.Popen(command, cwd=root, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    try:
        result = process.wait(timeout=timeout_minutes * 60)
    except subprocess.TimeoutExpired:
        try: os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError: pass
        try: process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            try: os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError: pass
            process.wait()
        report.mkdir(exist_ok=True)
        diagnostic = report / 'diagnostic.json'
        state = json.loads(diagnostic.read_text()) if diagnostic.exists() else {'schema': 1}
        state.update(status='error', supervisor_error=f'Full-model smoke exceeded {timeout_minutes} minutes')
        diagnostic.write_text(json.dumps(state, indent=2))
        result = process.returncode or 1
print('Report:', report)
print('Log:', log_path)
print('Exit:', result)
diagnostic = report / 'diagnostic.json'
if diagnostic.exists():
    state = json.loads(diagnostic.read_text())
    print('Status:', state.get('status'), 'Phase:', state.get('phase'))
    print('Losses:', state.get('losses'), 'Device memory:', state.get('memory_info'))
    print('Checkpoint:', state.get('checkpoint'))
print('\n'.join(log_path.read_text(errors='replace').splitlines()[-60:]))
if result: raise RuntimeError('Full-model TPU smoke failed; inspect/download diagnostic.json and the log')
